# Temporal Understanding Through fMRI Analysis

Testing the hypothesis that temporal understanding requires physiological state mediation (A->B->C).

## Hypothesis:
World models cannot gain comprehensive temporal understanding solely from external observations.
Biometric data provides insight into human temporal understanding through physiological state transitions.

In [ ]:
# Imports
%matplotlib inline
import torch
import wandb
from pathlib import Path

from learnedSpectrum.config import Config, DataConfig
from learnedSpectrum.models import VisionTransformerModel
from learnedSpectrum.train import train_loop, evaluate
from learnedSpectrum.visualization import TemporalUnderstandingVisualizer
from learnedSpectrum.physiological import PhysiologicalStateTracker
from learnedSpectrum.causal import CausalAnalysisModule
from learnedSpectrum.data import DatasetManager, create_dataloaders

In [ ]:
# Configuration
config = Config()
data_config = DataConfig()

In [ ]:
# Initialize wandb
wandb.init(
    project="temporal-understanding",
    config={
        "architecture": "vit-temporal",
        "dataset": "fmri-learning-stages",
        "epochs": config.NUM_EPOCHS,
        "batch_size": config.BATCH_SIZE,
        "learning_rate": config.LEARNING_RATE,
        "temporal_analysis": config.TEMPORAL_ANALYSIS,
        "causal_inference": config.CAUSAL_INFERENCE,
    },
)

In [ ]:
# Device setup and model initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Initialize models
model = VisionTransformerModel(config).to(device)
physiological_tracker = PhysiologicalStateTracker(config.EMBED_DIM).to(device)
causal_analyzer = CausalAnalysisModule(config.EMBED_DIM, config.TEMPORAL_DIM).to(device)

In [ ]:
# Initialize optimizer with gradient clipping
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.NUM_EPOCHS
)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
# Initialize visualization
vis_manager = TemporalUnderstandingVisualizer(config.VIS_DIR)

In [ ]:
# Load and preprocess data
data_manager = DatasetManager(config, data_config)
train_ds, val_ds, test_ds = data_manager.prepare_datasets()
train_loader, val_loader, test_loader = create_dataloaders(
    train_ds, val_ds, test_ds, config
)

In [ ]:
# Training loop
model = train_loop(
    model=model,
    train_dl=train_loader,
    val_dl=val_loader,
    optimizer=optimizer,
    config=config,
)

In [ ]:
# Evaluate and visualize results
test_loss, test_metrics = evaluate(
    model=model,
    test_dl=test_loader,
    config=config,
    physiological_tracker=physiological_tracker,
    causal_analyzer=causal_analyzer,
)

In [ ]:
# Plot comprehensive results
vis_manager.plot_state_transitions(
    test_metrics["state_analysis"], save_name="final_state_transitions"
)

vis_manager.plot_temporal_understanding_analysis(
    test_metrics["temporal_results"], save_name="final_temporal_analysis"
)

In [ ]:
# Log final results to wandb
wandb.log(
    {
        "test_accuracy": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "test_temporal_consistency": test_metrics["temporal_consistency"],
        "causal_strength": test_metrics["causal_analysis"]["mean_strength"],
        "ltc_stability": test_metrics["ltc_analysis"]["stability_metric"],
        "temporal_understanding_score": test_metrics["temporal_understanding_score"],
    }
)

In [ ]:
# Save visualizations to wandb
wandb.log(
    {
        "state_transitions": wandb.Image(
            str(Path(config.VIS_DIR) / "final_state_transitions.png")
        ),
        "temporal_analysis": wandb.Image(
            str(Path(config.VIS_DIR) / "final_temporal_analysis.png")
        ),
    }
)